# 04. Grad-CAM

학습된 ResNet18이 crop 이미지의 어느 영역을 보고 판단하는지 Grad-CAM으로 확인합니다.

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import datasets, models, transforms
ROOT = Path('C:/NEU-DET')
CROPS_DIR = ROOT / 'crops'
OUTPUT_FIGURES = ROOT / 'outputs' / 'figures'
OUTPUT_MODELS = ROOT / 'outputs' / 'models'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
checkpoint = torch.load(OUTPUT_MODELS / 'resnet18_best.pth', map_location=device)
class_names = checkpoint['class_names']
num_classes = len(class_names)
IMG_SIZE = checkpoint.get('img_size', 224)
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device).eval()
tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_ds = datasets.ImageFolder(CROPS_DIR / 'validation', transform=tfms)

In [ ]:
class GradCAM:
    """ResNet 마지막 convolution block의 activation과 gradient로 heatmap을 만듭니다."""
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.fwd_handle = target_layer.register_forward_hook(self._save_activation)
        self.bwd_handle = target_layer.register_full_backward_hook(self._save_gradient)
    def _save_activation(self, module, inputs, output):
        self.activations = output.detach()
    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    def __call__(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = int(output.argmax(dim=1).item())
        output[:, class_idx].sum().backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1).squeeze()
        cam = torch.relu(cam).cpu().numpy()
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx
    def close(self):
        self.fwd_handle.remove(); self.bwd_handle.remove()
gradcam = GradCAM(model, model.layer4[-1])

In [ ]:
def overlay_cam(image_path, cam):
    original = Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    original_np = np.array(original)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = np.uint8(0.55 * original_np + 0.45 * heatmap)
    return original_np, overlay
samples = []
for class_idx, cls in enumerate(class_names):
    for path, target in val_ds.samples:
        if target == class_idx:
            samples.append((path, target)); break
fig, axes = plt.subplots(len(samples), 2, figsize=(7, 3 * len(samples)))
if len(samples) == 1:
    axes = np.array([axes])
for row_idx, (path, target) in enumerate(samples):
    image_tensor = tfms(Image.open(path)).unsqueeze(0).to(device)
    cam, pred_idx = gradcam(image_tensor)
    original_np, overlay = overlay_cam(path, cam)
    axes[row_idx, 0].imshow(original_np); axes[row_idx, 0].set_title(f'True: {class_names[target]}'); axes[row_idx, 0].axis('off')
    axes[row_idx, 1].imshow(overlay); axes[row_idx, 1].set_title(f'Grad-CAM / Pred: {class_names[pred_idx]}'); axes[row_idx, 1].axis('off')
plt.tight_layout(); plt.savefig(OUTPUT_FIGURES / 'gradcam_samples.png', dpi=150); plt.show()
gradcam.close()